# Homework 3: VP-SDE Diffusion

Реализация Variance-Preserving SDE диффузии на данных в форме звезды.

- **Часть 1** (оценка 3): VP-SDE с линейным расписанием
- **Часть 2** (оценка 4): + сравнение перевзвешивания лосса
- **Часть 3** (оценка 5): + сравнение линейного и косинусового расписания

In [ ]:
import math

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def generate_star(n_spikes=5, inner_radius=0.4, outer_radius=1.0, n_samples=1000, center=(0, 0)):
    points = []
    angle_step = np.pi / n_spikes

    vertices = []
    for i in range(2 * n_spikes):
        angle = i * angle_step
        radius = outer_radius if i % 2 == 0 else inner_radius

        x = radius * np.cos(angle) + center[0]
        y = radius * np.sin(angle) + center[1]
        vertices.append([x, y])
    vertices.append(vertices[0])

    vertices = np.array(vertices)
    sampled_points = []

    for i in range(len(vertices) - 1):
        start_point = vertices[i]
        end_point = vertices[i + 1]

        for t in np.linspace(0, 1, n_samples // (len(vertices) - 1)):
            point = (1 - t) * start_point + t * end_point
            sampled_points.append(point)

    return np.array(sampled_points)


s = generate_star(n_samples=5000)
plt.scatter(s[:, 0], s[:, 1], s=1)
plt.title('Target distribution (star)')
plt.axis('equal')
plt.show()

## Noise Schedules

VP-SDE определяется через $\beta(t)$:
$$dx = -\frac{1}{2}\beta(t)x\,dt + \sqrt{\beta(t)}\,dw$$

Переходное распределение: $x_t = \alpha(t) x_0 + \sigma(t) \varepsilon$, где $\alpha(t)^2 + \sigma(t)^2 = 1$.

**Линейное расписание:** $\beta(t) = \beta_{min} + (\beta_{max} - \beta_{min})t$

**Косинусовое расписание:** $\alpha(t) = \cos(\pi t / 2)$, $\sigma(t) = \sin(\pi t / 2)$

In [ ]:
def get_alpha_sigma_linear(t, beta_min=0.1, beta_max=20.0):
    log_mean_coeff = -0.5 * (beta_min * t + 0.5 * (beta_max - beta_min) * t ** 2)
    alpha = torch.exp(log_mean_coeff)
    sigma = torch.sqrt(1.0 - alpha ** 2)
    return alpha, sigma


def get_alpha_sigma_cosine(t, **kwargs):
    alpha = torch.cos(math.pi * t / 2)
    sigma = torch.sin(math.pi * t / 2)
    return alpha, sigma


def get_beta_linear(t, beta_min=0.1, beta_max=20.0):
    return beta_min + (beta_max - beta_min) * t


def get_beta_cosine(t, **kwargs):
    alpha, sigma = get_alpha_sigma_cosine(t)
    # beta(t) = -2 * d/dt log(alpha(t)) = pi * tan(pi*t/2)
    return math.pi * sigma / (alpha + 1e-8)


SCHEDULES = {
    'linear': (get_alpha_sigma_linear, get_beta_linear),
    'cosine': (get_alpha_sigma_cosine, get_beta_cosine),
}

In [ ]:
# Визуализация расписаний
t_vis = torch.linspace(0, 1, 200)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for name in ['linear', 'cosine']:
    get_as, get_b = SCHEDULES[name]
    alpha, sigma = get_as(t_vis)
    axes[0].plot(t_vis.numpy(), alpha.numpy(), label=f'{name} alpha')
    axes[1].plot(t_vis.numpy(), sigma.numpy(), label=f'{name} sigma')
    snr = (alpha / (sigma + 1e-8)) ** 2
    axes[2].plot(t_vis.numpy(), torch.log10(snr).numpy(), label=f'{name} log10(SNR)')

for ax in axes:
    ax.legend()
    ax.grid(True)

axes[0].set_title('alpha(t)')
axes[1].set_title('sigma(t)')
axes[2].set_title('log10(SNR(t))')
plt.tight_layout()
plt.show()

## Перевзвешивание лосса

Лосс VP-SDE: $\mathcal{L} = \mathbb{E}_{t, x_0, \varepsilon}\left[ w(t) \|\varepsilon_\theta(x_t, t) - \varepsilon\|^2 \right]$

Варианты весов $w(t)$:
1. **Uniform**: $w(t) = 1$
2. **SNR**: $w(t) = \text{SNR}(t) = \alpha(t)^2 / \sigma(t)^2$ — больше внимания чистым шагам
3. **Truncated SNR (min-SNR-5)**: $w(t) = \min(\text{SNR}(t), 5)$ — обрезанный SNR
4. **Inverse SNR**: $w(t) = 1 / \text{SNR}(t) = \sigma(t)^2 / \alpha(t)^2$ — больше внимания шумным шагам

In [ ]:
def get_loss_weight(alpha, sigma, weighting='uniform'):
    snr = (alpha / (sigma + 1e-8)) ** 2
    if weighting == 'uniform':
        return torch.ones_like(alpha)
    elif weighting == 'snr':
        return snr
    elif weighting == 'min_snr_5':
        return torch.clamp(snr, max=5.0)
    elif weighting == 'inverse_snr':
        return 1.0 / (snr + 1e-8)
    else:
        raise ValueError(f'Unknown weighting: {weighting}')

## Модель и тренировка VP-SDE

In [ ]:
class VPSDEModel(nn.Module):
    def __init__(self, hidden_dim=256):
        super().__init__()
        self.time_embed = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.net = nn.Sequential(
            nn.Linear(2 + hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 2)
        )

    def forward(self, x, t):
        t_embed = self.time_embed(t)
        return self.net(torch.cat([x, t_embed], dim=1))

In [ ]:
class VPSDETrainer:
    def __init__(self, schedule='linear', weighting='uniform', beta_min=0.1, beta_max=20.0,
                 hidden_dim=256, lr=1e-3, batch_size=512, data_samples=5000):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.schedule = schedule
        self.weighting = weighting
        self.beta_min = beta_min
        self.beta_max = beta_max

        self.get_alpha_sigma, self.get_beta = SCHEDULES[schedule]

        star = generate_star(n_samples=data_samples)
        self.data = torch.tensor(star, dtype=torch.float32)
        self.dataloader = DataLoader(TensorDataset(self.data), batch_size=batch_size, shuffle=True)

        self.model = VPSDEModel(hidden_dim).to(self.device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)
        self.losses = []

    def train_epoch(self):
        self.model.train()
        total_loss = 0.0

        for (x0,) in self.dataloader:
            x0 = x0.to(self.device)
            eps = torch.randn_like(x0)

            t = torch.rand(x0.shape[0], 1, device=self.device)
            t = t.clamp(1e-5, 1.0 - 1e-5)

            alpha, sigma = self.get_alpha_sigma(t, beta_min=self.beta_min, beta_max=self.beta_max)
            x_t = alpha * x0 + sigma * eps

            pred_eps = self.model(x_t, t)

            w = get_loss_weight(alpha, sigma, self.weighting)
            loss = (w * (pred_eps - eps) ** 2).mean()

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

            total_loss += loss.item() * x0.shape[0]

        return total_loss / len(self.data)

    def run(self, epochs=3000):
        for n in range(epochs):
            loss = self.train_epoch()
            self.losses.append(loss)
            if n % 500 == 0:
                print(f'Epoch {n}/{epochs}, Loss: {loss:.6f}')

In [ ]:
@torch.no_grad()
def sample_vpsde(model, schedule='linear', beta_min=0.1, beta_max=20.0,
                 num_steps=1000, num_samples=1000):
    device = next(model.parameters()).device
    model.eval()

    get_alpha_sigma, get_beta = SCHEDULES[schedule]

    x = torch.randn(num_samples, 2, device=device)
    dt = 1.0 / num_steps

    for i in range(num_steps):
        t_val = 1.0 - i * dt
        t_tensor = torch.full((num_samples, 1), t_val, device=device)

        alpha, sigma = get_alpha_sigma(t_tensor, beta_min=beta_min, beta_max=beta_max)
        beta = get_beta(t_tensor, beta_min=beta_min, beta_max=beta_max)

        eps_pred = model(x, t_tensor)

        # Probability flow ODE: dx/dt = f(t)*x - 0.5*g(t)^2 * eps/sigma
        # f(t) = -0.5*beta(t), g(t)^2 = beta(t)
        # We go backward: x_{t-dt} = x_t - dt * [f(t)*x_t - 0.5*beta(t)*eps/sigma(t)]
        drift = -0.5 * beta * x - 0.5 * beta * eps_pred / (sigma + 1e-8)
        x = x - dt * drift

    return x.cpu().numpy()

## Часть 1: VP-SDE (оценка 3)

Базовая VP-SDE с линейным расписанием и uniform лоссом.

In [ ]:
trainer_base = VPSDETrainer(schedule='linear', weighting='uniform')
trainer_base.run(epochs=3000)

In [ ]:
samples_base = sample_vpsde(trainer_base.model, schedule='linear')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(s[:, 0], s[:, 1], s=1)
axes[0].set_title('Ground truth')
axes[0].axis('equal')
axes[1].scatter(samples_base[:, 0], samples_base[:, 1], s=1)
axes[1].set_title('VP-SDE (linear schedule, uniform loss)')
axes[1].axis('equal')
plt.tight_layout()
plt.show()

## Часть 2: Перевзвешивание лосса (оценка 4)

Сравниваем 4 варианта перевзвешивания лосса при линейном расписании:
1. Uniform: $w(t) = 1$
2. SNR: $w(t) = \alpha^2/\sigma^2$
3. Min-SNR-5: $w(t) = \min(\text{SNR}, 5)$
4. Inverse SNR: $w(t) = \sigma^2/\alpha^2$

In [ ]:
weightings = ['uniform', 'snr', 'min_snr_5', 'inverse_snr']
weighting_labels = ['Uniform', 'SNR', 'Min-SNR-5', 'Inverse SNR']

trainers_w = {}
trainers_w['uniform'] = trainer_base  # уже обучен

for w_name in weightings[1:]:
    print(f'\n=== Training with weighting: {w_name} ===')
    tr = VPSDETrainer(schedule='linear', weighting=w_name)
    tr.run(epochs=3000)
    trainers_w[w_name] = tr

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

for idx, (w_name, w_label) in enumerate(zip(weightings, weighting_labels)):
    ax = axes[idx // 2][idx % 2]
    samp = sample_vpsde(trainers_w[w_name].model, schedule='linear')
    ax.scatter(samp[:, 0], samp[:, 1], s=1, alpha=0.5)
    ax.set_title(f'Linear + {w_label}', fontsize=14)
    ax.axis('equal')
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)

plt.suptitle('Сравнение перевзвешивания лосса (linear schedule)', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for w_name, w_label in zip(weightings, weighting_labels):
    losses = trainers_w[w_name].losses
    ax.plot(losses, label=w_label, alpha=0.7)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training loss (linear schedule, different weightings)')
ax.legend()
ax.set_yscale('log')
ax.grid(True)
plt.tight_layout()
plt.show()

### Вывод по перевзвешиванию лосса

- **Uniform** ($w=1$) — базовый вариант. Модель уделяет одинаковое внимание всем уровням шума. Обычно даёт стабильный, но не оптимальный результат.
- **SNR** ($w=\alpha^2/\sigma^2$) — акцент на малозашумлённых шагах (около $t=0$). Это помогает модели лучше восстанавливать мелкие детали звезды (острые кончики), но может недоучиться на шумных шагах.
- **Min-SNR-5** — компромиссный вариант: даёт преимущество чистым шагам, но ограничивает вес сверху. Обычно лучше всего передаёт общую форму и детали.
- **Inverse SNR** ($w=\sigma^2/\alpha^2$) — акцент на шумных шагах. Модель хорошо учится денойзить из сильного шума (правильная общая структура), но может хуже передавать мелкие детали.

**Лучший вариант для звёздочки — Min-SNR-5**, так как он балансирует между общей формой и деталями.

## Часть 3: Сравнение расписаний (оценка 5)

Сравниваем линейное и косинусовое расписание со всеми вариантами перевзвешивания.

In [ ]:
trainers_cosine = {}

for w_name in weightings:
    print(f'\n=== Training cosine + {w_name} ===')
    tr = VPSDETrainer(schedule='cosine', weighting=w_name)
    tr.run(epochs=3000)
    trainers_cosine[w_name] = tr

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for col, (w_name, w_label) in enumerate(zip(weightings, weighting_labels)):
    # Linear
    samp_lin = sample_vpsde(trainers_w[w_name].model, schedule='linear')
    axes[0][col].scatter(samp_lin[:, 0], samp_lin[:, 1], s=1, alpha=0.5)
    axes[0][col].set_title(f'Linear + {w_label}', fontsize=11)
    axes[0][col].axis('equal')
    axes[0][col].set_xlim(-1.5, 1.5)
    axes[0][col].set_ylim(-1.5, 1.5)

    # Cosine
    samp_cos = sample_vpsde(trainers_cosine[w_name].model, schedule='cosine')
    axes[1][col].scatter(samp_cos[:, 0], samp_cos[:, 1], s=1, alpha=0.5)
    axes[1][col].set_title(f'Cosine + {w_label}', fontsize=11)
    axes[1][col].axis('equal')
    axes[1][col].set_xlim(-1.5, 1.5)
    axes[1][col].set_ylim(-1.5, 1.5)

axes[0][0].set_ylabel('Linear schedule', fontsize=13)
axes[1][0].set_ylabel('Cosine schedule', fontsize=13)

plt.suptitle('Сравнение расписаний и перевзвешивания лосса (2 schedules × 4 weightings)', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for w_name, w_label in zip(weightings, weighting_labels):
    axes[0].plot(trainers_w[w_name].losses, label=w_label, alpha=0.7)
axes[0].set_title('Linear schedule')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].set_yscale('log')
axes[0].grid(True)

for w_name, w_label in zip(weightings, weighting_labels):
    axes[1].plot(trainers_cosine[w_name].losses, label=w_label, alpha=0.7)
axes[1].set_title('Cosine schedule')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].set_yscale('log')
axes[1].grid(True)

plt.suptitle('Training loss comparison', fontsize=14)
plt.tight_layout()
plt.show()

### Выводы

#### Сравнение расписаний

- **Линейное расписание** быстро переводит данные в шум (SNR резко падает), что означает, что модель большую часть времени работает в сильно зашумлённом режиме. Из-за этого ей сложнее восстанавливать мелкие детали.
- **Косинусовое расписание** более плавно распределяет уровень шума по времени — SNR снижается равномернее. Благодаря этому модель получает больше обучающего сигнала на промежуточных уровнях шума, что обычно приводит к лучшему качеству генерации.

#### Сравнение перевзвешивания

- **Uniform** — базовый вариант, одинаковое внимание всем $t$.
- **SNR** — больше внимания малозашумлённым шагам → лучше детали, но может страдать общая форма.
- **Min-SNR-5** — лучший компромисс: помогает с деталями, но не забывает про шумные шаги.
- **Inverse SNR** — акцент на денойзинг из сильного шума → хорошая общая форма, слабые детали.

#### Итого

Лучшая комбинация — **косинусовое расписание + Min-SNR-5**: косинусовое расписание обеспечивает плавный переход, а Min-SNR-5 правильно балансирует внимание модели между разными уровнями шума.